In [1]:
# ================================================================
#  LCCN-R25 - TRAIN/TEST NEAR-DUPLICATE AUDIT  [CPU, ~10 min]
#  Answers IJIES Reviewer 2, point 5 (split contamination).
#  Part A: for every test comment, the maximum character-5-gram
#          Jaccard similarity to any training comment (exact, all
#          6,000 x 28,000 pairs, chunked sparse matmul), plus exact
#          match after lowercasing. Counts at Jaccard >= 0.8 and
#          >= 0.9. Same audit for val vs train and test vs val.
#  Part B: re-scores the STORED XLM-RoBERTa predictions (nb4 seed 42,
#          nb6 seed 44) on the test subset with contaminated comments
#          removed, so no retraining is needed.
#  Part C: reports which metadata columns exist in the corpus file,
#          to decide whether a temporal or source split is possible.
#  Add Input: paper1-nb1-lccn-final (output: splits/), final-emotion-
#             dataset, paper1-nb4-lccn-xlmroberta (output),
#             paper1-nb6-lccn-xlm43-44 (output).
#  Accelerator None. Save & Run All (Commit). Transcribe ONLY from log.
# ================================================================
import glob, os, re, sys, time
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import f1_score
print("python", sys.version.split()[0])

sd = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/splits/train.csv", recursive=True)]
assert sd, "attach the paper1-nb1-lccn-final output (splits/)"
def load_split(n):
    d = pd.read_csv(f"{sd[0]}/{n}.csv", dtype=str, keep_default_na=False)
    d.columns = [c.strip().lower() for c in d.columns]
    idc = next(c for c in d.columns if c in ("id", "comment_id"))
    lab = next(c for c in ("majority", "gold_label", "label") if c in d.columns)
    return d.rename(columns={idc: "id", lab: "label"})
tr, va, te = load_split("train"), load_split("val"), load_split("test")
assert (len(tr), len(va), len(te)) == (28000, 6000, 6000), (len(tr), len(va), len(te))
print(f"train {len(tr):,} | val {len(va):,} | test {len(te):,}")
print("columns:", list(tr.columns)[:8], "...")


python 3.12.13
train 28,000 | val 6,000 | test 6,000
columns: ['id', 'comment', 'label', 'text_raw', 'text_keel', 'text_kope', 'text_metaphone', 'text_terun'] ...


In [2]:
# ================================================================
#  CELL 1 - PART A: EXACT MAX-JACCARD OVER CHARACTER 5-GRAMS
# ================================================================
def norm(s): return re.sub(r"\s+", " ", str(s).lower()).strip()
for d in (tr, va, te): d["t"] = d["text_raw"].map(norm)

vec = CountVectorizer(analyzer="char", ngram_range=(5, 5), binary=True, lowercase=False)
vec.fit(pd.concat([tr.t, va.t, te.t]))
def mat(d): return vec.transform(d.t).astype(np.int32)
Xtr, Xva, Xte = mat(tr), mat(va), mat(te)
ntr, nva, nte = np.asarray(Xtr.sum(1)).ravel(), np.asarray(Xva.sum(1)).ravel(), np.asarray(Xte.sum(1)).ravel()
print(f"5-gram vocabulary {len(vec.vocabulary_):,} | mean 5-grams per comment "
      f"train {ntr.mean():.1f} val {nva.mean():.1f} test {nte.mean():.1f}")

def max_jaccard(XA, nA, XB, nB, chunk=400):
    """for each row of A, the max Jaccard to any row of B, and that row's index"""
    best = np.zeros(XA.shape[0]); arg = np.zeros(XA.shape[0], dtype=int)
    XBt = XB.T.tocsc()
    for s in range(0, XA.shape[0], chunk):
        inter = (XA[s:s+chunk] @ XBt).toarray().astype(np.float64)
        union = nA[s:s+chunk, None] + nB[None, :] - inter
        jac = np.where(union > 0, inter / np.maximum(union, 1), 0.0)
        arg[s:s+chunk] = jac.argmax(1); best[s:s+chunk] = jac.max(1)
    return best, arg

t0 = time.time()
te["jac_train"], te["nn_train"] = max_jaccard(Xte, nte, Xtr, ntr)
va["jac_train"], va["nn_train"] = max_jaccard(Xva, nva, Xtr, ntr)
te["jac_val"],   te["nn_val"]   = max_jaccard(Xte, nte, Xva, nva)
print(f"pairwise Jaccard done in {time.time()-t0:.0f}s")

train_texts = set(tr.t)
te["exact_train"] = te.t.isin(train_texts); va["exact_train"] = va.t.isin(train_texts)

def report(d, col, name):
    print(f"\n{name}: n = {len(d):,}")
    for thr in (0.5, 0.6, 0.7, 0.8, 0.9, 1.0):
        k = int((d[col] >= thr).sum()); print(f"  Jaccard >= {thr:.1f}: {k:>5,}  ({100*k/len(d):5.2f}%)")
    if "exact_train" in d and col == "jac_train":
        k = int(d.exact_train.sum()); print(f"  exact match after lowercasing: {k:,} ({100*k/len(d):.2f}%)")
    print(f"  mean max-Jaccard {d[col].mean():.3f} | median {d[col].median():.3f}")
report(te, "jac_train", "TEST vs TRAIN")
report(va, "jac_train", "VAL vs TRAIN")
report(te, "jac_val",   "TEST vs VAL")

# label agreement of near-duplicate pairs (does contamination carry the label?)
for thr in (0.8, 0.9):
    m = te.jac_train >= thr
    if m.sum():
        same = (te.loc[m, "label"].values == tr.label.values[te.loc[m, "nn_train"].values]).mean()
        print(f"\ntest-train pairs at Jaccard >= {thr}: {int(m.sum()):,}; same label in {100*same:.1f}% of them")

print("\nten highest-similarity test-train pairs:")
for _, r in te.sort_values("jac_train", ascending=False).head(10).iterrows():
    print(f"  {r.jac_train:.3f} | {r.label:<8} test: {r.t[:70]}")
    print(f"        | {tr.label.values[r.nn_train]:<8} train: {tr.t.values[r.nn_train][:70]}")

te[["id", "label", "jac_train", "nn_train", "jac_val", "exact_train"]].to_csv(
    "/kaggle/working/test_near_duplicates.csv", index=False)
for thr in (0.8, 0.9):
    te.loc[te.jac_train >= thr, ["id"]].to_csv(f"/kaggle/working/contaminated_test_ids_j{int(thr*100)}.csv", index=False)
print("\nwritten: test_near_duplicates.csv, contaminated_test_ids_j80.csv, contaminated_test_ids_j90.csv")


5-gram vocabulary 309,180 | mean 5-grams per comment train 54.0 val 52.9 test 54.2
pairwise Jaccard done in 12s

TEST vs TRAIN: n = 6,000
  Jaccard >= 0.5:   740  (12.33%)
  Jaccard >= 0.6:   464  ( 7.73%)
  Jaccard >= 0.7:   292  ( 4.87%)
  Jaccard >= 0.8:   158  ( 2.63%)
  Jaccard >= 0.9:    79  ( 1.32%)
  Jaccard >= 1.0:    30  ( 0.50%)
  exact match after lowercasing: 21 (0.35%)
  mean max-Jaccard 0.260 | median 0.197

VAL vs TRAIN: n = 6,000
  Jaccard >= 0.5:   702  (11.70%)
  Jaccard >= 0.6:   431  ( 7.18%)
  Jaccard >= 0.7:   272  ( 4.53%)
  Jaccard >= 0.8:   167  ( 2.78%)
  Jaccard >= 0.9:    70  ( 1.17%)
  Jaccard >= 1.0:    20  ( 0.33%)
  exact match after lowercasing: 9 (0.15%)
  mean max-Jaccard 0.257 | median 0.194

TEST vs VAL: n = 6,000
  Jaccard >= 0.5:   296  ( 4.93%)
  Jaccard >= 0.6:   175  ( 2.92%)
  Jaccard >= 0.7:   110  ( 1.83%)
  Jaccard >= 0.8:    54  ( 0.90%)
  Jaccard >= 0.9:    20  ( 0.33%)
  Jaccard >= 1.0:     3  ( 0.05%)
  mean max-Jaccard 0.185 | median 

In [3]:
# ================================================================
#  CELL 2 - PART B: RE-SCORE STORED XLM-RoBERTa PREDICTIONS ON THE
#  DEDUPLICATED TEST SUBSETS (no retraining)
#  nb4 preds = seed 42 (all arms); nb6 preds = seed 44 (raw, indicsoundex, lccn);
#  any __seed{N}.csv file = seed N.
# ================================================================
MODEL = "XLM-RoBERTa"
def seed_of(p):
    m = re.search(r"__seed(\d+)\.csv$", p)
    if m: return int(m.group(1))
    if "nb4-lccn-xlmroberta" in p: return 42
    if "nb6-lccn-xlm43-44" in p: return 44
    return None
preds = {}
for f in sorted(glob.glob(f"/kaggle/input/**/preds/{MODEL}__*.csv", recursive=True)):
    arm = re.sub(r"__seed\d+$", "", os.path.basename(f)[len(MODEL)+2:-4]); s = seed_of(f)
    if s is not None: preds[(arm, s)] = f
print(f"{len(preds)} stored (arm, seed) prediction sets")

def load_pred(p):
    d = pd.read_csv(p, dtype=str, keep_default_na=False); d.columns = [c.strip().lower() for c in d.columns]
    idc = next(c for c in d.columns if c in ("id", "comment_id"))
    y = next(c for c in d.columns if "true" in c or c == "gold"); pr = next(c for c in d.columns if "pred" in c)
    d = d[[idc, y, pr]].copy(); d.columns = ["id", "y", "p"]
    for c in ("y", "p"): d[c] = d[c].str.strip().str.lower()
    return d.set_index("id")

keep = {"full": set(te.id),
        "drop_j80": set(te.loc[te.jac_train < 0.8, "id"]),
        "drop_j90": set(te.loc[te.jac_train < 0.9, "id"])}
print({k: len(v) for k, v in keep.items()})

rows = []
for (arm, s), f in sorted(preds.items(), key=lambda kv: (kv[0][1], kv[0][0])):
    d = load_pred(f); assert len(d) == 6000, (arm, s, len(d))
    row = dict(arm=arm, seed=s)
    for k, ids in keep.items():
        sub = d[d.index.isin(ids)]
        row[k] = round(f1_score(sub.y, sub.p, average="macro"), 4)
    row["delta_j80"] = round(row["drop_j80"] - row["full"], 4)
    row["delta_j90"] = round(row["drop_j90"] - row["full"], 4)
    rows.append(row)
sub = pd.DataFrame(rows)
sub.to_csv("/kaggle/working/dedup_subset_macro_f1.csv", index=False)
print("\n=== XLM-RoBERTa macro-F1: full test vs deduplicated subsets ===")
print(sub.to_string(index=False))

print("\n=== ordering check (raw, lccn, lccn_indicxlit) per seed ===")
for s in sorted({x[1] for x in preds}):
    g = sub[sub.seed == s].set_index("arm")
    for k in ("full", "drop_j80", "drop_j90"):
        order = [a for a in ("raw", "indicsoundex", "lccn", "lccn_base", "lccn_indicxlit") if a in g.index]
        print(f"  seed {s} {k:<9} " + "  ".join(f"{a} {g.loc[a, k]:.4f}" for a in order))


16 stored (arm, seed) prediction sets
{'full': 6000, 'drop_j80': 5842, 'drop_j90': 5921}

=== XLM-RoBERTa macro-F1: full test vs deduplicated subsets ===
               arm  seed   full  drop_j80  drop_j90  delta_j80  delta_j90
      indicsoundex    42 0.7773    0.7739    0.7751    -0.0034    -0.0022
indicsoundex_guard    42 0.7861    0.7829    0.7837    -0.0032    -0.0024
              keel    42 0.8004    0.7971    0.7982    -0.0033    -0.0022
              kope    42 0.8090    0.8063    0.8073    -0.0027    -0.0017
              lccn    42 0.7991    0.7956    0.7969    -0.0035    -0.0022
         lccn_base    42 0.7991    0.7968    0.7975    -0.0023    -0.0016
    lccn_indicxlit    42 0.8130    0.8100    0.8110    -0.0030    -0.0020
       lccn_itrans    42 0.8051    0.8019    0.8031    -0.0032    -0.0020
         metaphone    42 0.7916    0.7880    0.7893    -0.0036    -0.0023
               raw    42 0.8058    0.8035    0.8043    -0.0023    -0.0015
           soundex    42 0.7786 

In [4]:
# ================================================================
#  CELL 3 - PART C: METADATA AVAILABLE FOR A GROUPED OR TEMPORAL SPLIT,
#  AND THE VERDICT
# ================================================================
gold = [f for f in glob.glob("/kaggle/input/**/Gold_dataset*.csv", recursive=True)]
if gold:
    g = pd.read_csv(gold[0], dtype=str, keep_default_na=False, nrows=5)
    print("corpus file columns:", list(g.columns))
    meta = [c for c in g.columns if any(k in c.lower() for k in
            ("date", "time", "video", "film", "movie", "source", "url", "channel", "year"))]
    print("metadata usable for a grouped/temporal split:", meta if meta else "NONE")
else:
    print("Gold_dataset.csv not attached; cannot report metadata columns")

n80, n90 = int((te.jac_train >= 0.8).sum()), int((te.jac_train >= 0.9).sum())
nx = int(te.exact_train.sum())
print("\nVERDICT (numbers for Section 3.4 and the Conclusion)")
print(f"  test comments with a training near-duplicate at Jaccard >= 0.8: {n80:,} ({100*n80/6000:.2f}%)")
print(f"  test comments with a training near-duplicate at Jaccard >= 0.9: {n90:,} ({100*n90/6000:.2f}%)")
print(f"  exact duplicates of a training comment (lowercased):            {nx:,} ({100*nx/6000:.2f}%)")
if len(sub):
    worst = sub.delta_j80.abs().max()
    print(f"  largest |change| in XLM-RoBERTa macro-F1 after removing j>=0.8 duplicates: {worst:.4f}")
print("\nDONE - test_near_duplicates.csv, contaminated_test_ids_j80/j90.csv, dedup_subset_macro_f1.csv written.")
print("Save & Run All (Commit); transcribe only from the committed log.")


corpus file columns: ['id', 'comment', 'ae1', 'ae2', 'ae3', 'majority']
metadata usable for a grouped/temporal split: NONE

VERDICT (numbers for Section 3.4 and the Conclusion)
  test comments with a training near-duplicate at Jaccard >= 0.8: 158 (2.63%)
  test comments with a training near-duplicate at Jaccard >= 0.9: 79 (1.32%)
  exact duplicates of a training comment (lowercased):            21 (0.35%)
  largest |change| in XLM-RoBERTa macro-F1 after removing j>=0.8 duplicates: 0.0036

DONE - test_near_duplicates.csv, contaminated_test_ids_j80/j90.csv, dedup_subset_macro_f1.csv written.
Save & Run All (Commit); transcribe only from the committed log.
